[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/52_context_memory_solution.ipynb)

#  Solution: Context Window Memory Manager

Reference: MemGPT (Packer et al. 2023), Letta framework, and standard LLM context management.

```text
TRUNCATE: remove oldest user/assistant messages first, keep system message
SLIDING: keep last N tokens worth of messages
SUMMARIZE: compress old messages into a tight summary label
```

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  SOLUTION

def manage_context(history, new_observation, max_tokens=4096,
                  token_counter=None, strategy="sliding"):
    """Manage agent context memory.

    Inspired by MemGPT/Letta: OS-inspired virtual memory for LLM context.
    Three strategies map to real-world approaches:
    - TRUNCATE: simple FIFO, drops oldest first (like C4/FineWeb filtering)
    - SLIDING: keep most recent tokens (like ChatGPT's context window)
    - SUMMARIZE: compression (like MemGPT's memory hierarchy)
    """
    if token_counter is None:
        token_counter = len
    remaining = max_tokens - token_counter(new_observation)
    result = []

    if strategy == "truncate":
        for msg in history:
            if msg["role"] == "system":
                result.append(msg)
                remaining -= token_counter(msg["content"])
                break
        for msg in reversed(history):
            if msg["role"] == "system": continue
            cost = token_counter(msg["content"])
            if remaining >= cost:
                insert_pos = 1 if result and result[0]["role"] == "system" else 0
                result.insert(insert_pos, msg)
                remaining -= cost

    elif strategy == "sliding":
        for msg in reversed(history):
            cost = token_counter(msg["content"])
            if remaining >= cost:
                result.insert(0, msg)
                remaining -= cost
            else:
                break

    elif strategy == "summarize":
        kept = []
        oldest = []
        for msg in history:
            cost = token_counter(msg["content"])
            if remaining >= cost:
                kept.append(msg)
                remaining -= cost
            else:
                oldest.append(msg)
        if oldest:
            summary_content = f"[{len(oldest)} prior msgs]"
            if token_counter(summary_content) > remaining and kept:
                while kept and token_counter(summary_content) > remaining:
                    m = kept.pop(0)
                    oldest.insert(0, m)
                    remaining += token_counter(m["content"])
                summary_content = f"[{len(oldest)} prior msgs]"
            result = [{"role": "summary", "content": summary_content}] + kept
        else:
            result = kept

    if new_observation:
        result.append({"role": "observation", "content": new_observation})
    return result

In [ ]:
#  Verify
h = [
    {"role": "system", "content": "Bot"},
    {"role": "user", "content": "Hi" * 20},
    {"role": "assistant", "content": "Hello" * 20},
]
result = manage_context(h, "OK", max_tokens=60, token_counter=len, strategy="sliding")
print(f"Kept {len(result)} messages, total tokens: {sum(len(m['content']) for m in result)}")

In [ ]:
from torch_judge import check
check("context_memory")